<a href="https://colab.research.google.com/github/bmanibharathibe/trendpulse-manibharathi/blob/main/Data_Analys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import requests
import json
import os
import time
from datetime import datetime

headers = {"User-Agent": "TrendPulse/1.0"}

TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

categories = {
    "technology": ["ai","software","tech","code","computer","data","cloud","api","gpu","llm"],
    "worldnews": ["war","government","country","president","election","climate","attack","global"],
    "sports": ["nfl","nba","fifa","sport","game","team","player","league","championship"],
    "science": ["research","study","space","physics","biology","discovery","nasa","genome"],
    "entertainment": ["movie","film","music","netflix","game","book","show","award","streaming"]
}

collected_stories = []
category_counts = {cat: 0 for cat in categories}

# Fetch top story IDs
try:
    response = requests.get(TOP_STORIES_URL, headers=headers)
    story_ids = response.json()
except:
    print("Error fetching top stories")
    story_ids = []

# Function to detect category
def detect_category(title):
    title = title.lower()
    for cat, words in categories.items():
        for word in words:
            if word in title:
                return cat
    return None

# Loop through categories
for category in categories:

    print(f"Collecting stories for {category}")

    for story_id in story_ids:

        if category_counts[category] >= 35:
            break

        try:
            r = requests.get(ITEM_URL.format(story_id), headers=headers)
            story = r.json()
        except:
            print("Request failed for", story_id)
            continue

        if not story or "title" not in story:
            continue

        detected = detect_category(story["title"])

        if detected != category:
            continue

        story_data = {
            "post_id": story.get("id"),
            "title": story.get("title"),
            "category": category,
            "score": story.get("score", 0),
            "num_comments": story.get("descendants", 0),
            "author": story.get("by"),
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        collected_stories.append(story_data)
        category_counts[category] += 1

    # sleep after each category
    time.sleep(2)

# Create data folder
if not os.path.exists("data"):
    os.makedirs("data")

filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

with open(filename, "w") as f:
    json.dump(collected_stories, f, indent=4)

print(f"Collected {len(collected_stories)} stories. Saved to {filename}")

Collected 112 stories. Saved to data/trends_20260408.json


In [5]:
import pandas as pd
import os
import json
import glob

# -------------------------------
# Step 1 — Load the JSON file
# -------------------------------

# Find the latest JSON file in the data folder
files = glob.glob("data/trends_*.json")

if not files:
    print("No JSON file found in data/ folder. Please run task1_data_collection.py first.")
    exit()

latest_file = max(files)

# Load JSON into pandas DataFrame
df = pd.read_json(latest_file)

print(f"Loaded {len(df)} stories from {latest_file}")

# -------------------------------
# Step 2 — Clean the data
# -------------------------------

# Remove duplicate stories based on post_id
before = len(df)
df = df.drop_duplicates(subset="post_id")
print(f"After removing duplicates: {len(df)}")

# Remove rows with missing values in key fields
df = df.dropna(subset=["post_id", "title", "score"])
print(f"After removing nulls: {len(df)}")

# Convert score and num_comments to integers
df["score"] = df["score"].astype(int)
df["num_comments"] = df["num_comments"].astype(int)

# Remove low-quality stories (score < 5)
df = df[df["score"] >= 5]
print(f"After removing low scores: {len(df)}")

# Remove extra whitespace from titles
df["title"] = df["title"].str.strip()

# -------------------------------
# Step 3 — Save cleaned CSV
# -------------------------------

output_file = "data/trends_clean.csv"
df.to_csv(output_file, index=False)

print(f"\nSaved {len(df)} rows to {output_file}")

# -------------------------------
# Summary: stories per category
# -------------------------------

print("\nStories per category:")

category_summary = df["category"].value_counts()

for category, count in category_summary.items():
    print(f"  {category:<15} {count}")

Loaded 112 stories from data/trends_20260408.json
After removing duplicates: 112
After removing nulls: 112
After removing low scores: 108

Saved 108 rows to data/trends_clean.csv

Stories per category:
  technology      34
  entertainment   33
  worldnews       21
  sports          14
  science         6


In [6]:
# task3_analysis.py
# TrendPulse - Task 3: Analysis with Pandas & NumPy

# Import required libraries
import pandas as pd
import numpy as np

# -----------------------------------------------------
# 1 — Load and Explore the Data
# -----------------------------------------------------

# Load the cleaned dataset from Task 2
df = pd.read_csv("data/trends_clean.csv")

# Print dataset shape (rows, columns)
print("Loaded data:", df.shape)

# Display first 5 rows
print("\nFirst 5 rows:")
print(df.head())

# Calculate average score and average number of comments
avg_score = df["score"].mean()
avg_comments = df["num_comments"].mean()

print("\nAverage score   :", round(avg_score, 2))
print("Average comments:", round(avg_comments, 2))


# -----------------------------------------------------
# 2 — Basic Analysis with NumPy
# -----------------------------------------------------

print("\n--- NumPy Stats ---")

# Convert pandas columns to NumPy arrays
scores = df["score"].to_numpy()
comments = df["num_comments"].to_numpy()

# Mean, median and standard deviation of score
mean_score = np.mean(scores)
median_score = np.median(scores)
std_score = np.std(scores)

print("Mean score   :", round(mean_score, 2))
print("Median score :", round(median_score, 2))
print("Std deviation:", round(std_score, 2))

# Highest and lowest score
print("Max score    :", np.max(scores))
print("Min score    :", np.min(scores))

# Category with the most stories
category_counts = df["category"].value_counts()
top_category = category_counts.idxmax()
top_category_count = category_counts.max()

print("\nMost stories in:", top_category, f"({top_category_count} stories)")

# Story with the most comments
max_comment_index = np.argmax(comments)
most_commented_story = df.iloc[max_comment_index]

print("\nMost commented story:")
print(f"\"{most_commented_story['title']}\" — {most_commented_story['num_comments']} comments")


# -----------------------------------------------------
# 3 — Add New Columns
# -----------------------------------------------------

# Engagement = comments divided by (score + 1)
# +1 prevents division by zero if score is 0
df["engagement"] = df["num_comments"] / (df["score"] + 1)

# A story is popular if its score is greater than the average score
df["is_popular"] = df["score"] > avg_score


# -----------------------------------------------------
# 4 — Save the Result
# -----------------------------------------------------

# Save the updated dataframe to a new CSV
output_path = "data/trends_analysed.csv"
df.to_csv(output_path, index=False)

print("\nSaved to", output_path)

Loaded data: (108, 7)

First 5 rows:
    post_id                                              title    category  \
0  47687273         Git commands I run before reading any code  technology   
1  47689174  MegaTrain: Full Precision Training of 100B+ Pa...  technology   
2  47689237  US cities are axing Flock Safety surveillance ...  technology   
3  47690415  Show HN: We fingerprinted 178 AI models' writi...  technology   
4  47679121  Project Glasswing: Securing critical software ...  technology   

   score  num_comments           author         collected_at  
0    807           178       grepsedawk  2026-04-08 14:57:34  
1    103            21            chrsw  2026-04-08 14:57:34  
2    198            99  giuliomagnifico  2026-04-08 14:57:34  
3     16             6        nuancedev  2026-04-08 14:57:34  
4   1406           728         Ryan5453  2026-04-08 14:57:34  

Average score   : 191.95
Average comments: 86.04

--- NumPy Stats ---
Mean score   : 191.95
Median score : 97.5
Std